# MyGram reproduction on Google Colab

This notebook restores a clean source bundle and verified PMF archive from Google Drive, validates CUDA/data, runs the required GPU gates, and resumes the 38-run matrix. Select a GPU runtime before running it. Upload `mygram-source.zip`, `source_bundle_manifest.json`, and `pmf_mmkg.zip` to the Drive paths used below.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/MyGramRepro')
BUNDLE = DRIVE_ROOT / 'source/mygram-source.zip'
BUNDLE_MANIFEST = DRIVE_ROOT / 'source/source_bundle_manifest.json'
PMF_ARCHIVE = DRIVE_ROOT / 'sources/pmf_mmkg.zip'
for required in (BUNDLE, BUNDLE_MANIFEST, PMF_ARCHIVE):
    if not required.is_file():
        raise FileNotFoundError(required)
print('Drive artifacts found:', DRIVE_ROOT)

In [ ]:
import hashlib, json, os, shutil, zipfile
manifest = json.loads(BUNDLE_MANIFEST.read_text(encoding='utf-8'))
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
if BUNDLE.stat().st_size != manifest['bundle']['bytes'] or sha256(BUNDLE) != manifest['bundle']['sha256']:
    raise ValueError('Source bundle does not match its manifest')
WORKSPACE = Path('/content/mygram')
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir()
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(WORKSPACE)
shutil.copy2(BUNDLE_MANIFEST, WORKSPACE / 'source_bundle_manifest.json')
os.chdir(WORKSPACE)
print('Source restored:', manifest['bundle']['sha256'])

In [ ]:
from src.notebook_runner import run_script
run_script(Path('scripts/notebook_steps/common/03_install_dependencies.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/common/04_check_environment.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/common/05_prepare_data.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/base/06_run_unit_tests.py'), globals())


## Mandatory CUDA gates
Run the next three cells in order. G3 and G4 profile full real graphs. G6 creates separate 60-epoch pilots and tests Drive-backed chunk resume. Their metrics are not research results.

In [ ]:
run_script(Path('scripts/notebook_steps/base/08_gate_g3.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/base/09_gate_g4.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/base/10_gate_g6.py'), globals())


## Run or resume the research matrix
The runner executes one experiment at a time, checkpoints locally every epoch, and publishes a verified Drive snapshot every five epochs. Re-run this cell in a later Colab session to resume. It stops before the configured wall-time budget.

In [ ]:
run_script(Path('scripts/notebook_steps/base/12_run_matrix.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/base/13_show_matrix_status.py'), globals())
